In [20]:
#@title Setup and Installation
# Install required packages
!pip install -q numpy pandas scikit-learn matplotlib seaborn lightgbm xgboost catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 15.0 MB/s eta 0:00:00


In [21]:
# @title Load Data
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
train = pd.read_csv('/content/drive/MyDrive/bt-4012-in-class-kaggle-competiton-1-2025/train.csv')
test = pd.read_csv('/content/drive/MyDrive/bt-4012-in-class-kaggle-competiton-1-2025/test.csv')
sample = pd.read_csv('/content/drive/MyDrive/bt-4012-in-class-kaggle-competiton-1-2025/sample_submission.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#@title Data Preprocessing
# Fill missing categorical/text values
text_cols = ["description", "location", "profile_image_url", "profile_background_image_url", "lang"]
for col in text_cols:
    train[col] = train[col].fillna("unknown")
    test[col] = test[col].fillna("unknown")

# Binary indicators for presence
for col in text_cols:
    feat_name = f"has_{col.split('_')[0]}"
    train[feat_name] = (train[col] != "unknown").astype(int)
    test[feat_name] = (test[col] != "unknown").astype(int)

# Fill missing numeric values
num_cols = ["followers_count", "friends_count", "favourites_count",
            "statuses_count", "average_tweets_per_day", "account_age_days"]
for col in num_cols:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

# Check if any NaNs remain
print("Missing in train:\n", train.isnull().sum().sort_values(ascending=False).head())
print("\nMissing in test:\n", test.isnull().sum().sort_values(ascending=False).head())

Missing values after imputation (Train):
created_at               0
default_profile          0
default_profile_image    0
description              0
favourites_count         0
followers_count          0
friends_count            0
geo_enabled              0
id                       0
lang                     0
dtype: int64

Missing values after imputation (Test):
index                    0
created_at               0
default_profile          0
default_profile_image    0
description              0
favourites_count         0
followers_count          0
friends_count            0
geo_enabled              0
id                       0
dtype: int64


In [19]:
#@title Feature Engineering
import numpy as np

# Copy base frame
train_fe = train.copy()
test_fe = test.copy()

# Utility functions (length, digits, ratios, etc.)
def safe_len(s): return len(str(s))
def frac_digits(s): return sum(c.isdigit() for c in str(s)) / (len(str(s)) + 1e-6)
def ends_with_digit(s): return str(s)[-1].isdigit() if str(s) else False
def count_upper(s): return sum(c.isupper() for c in str(s))
def count_special(s): return sum(c in set("_-.") for c in str(s))
def word_count(s): return len(str(s).split())
def avg_word_len(s): return np.mean([len(w) for w in str(s).split()]) if str(s).split() else 0
def special_char_ratio(s): return sum(not c.isalnum() for c in str(s)) / (len(str(s)) + 1)

# Log transforms for skewed numeric columns
for col in ["followers_count", "friends_count", "statuses_count", "favourites_count"]:
    train_fe[f"log1p_{col}"] = np.log1p(train_fe[col])
    test_fe[f"log1p_{col}"] = np.log1p(test_fe[col])

# Engagement ratios
train_fe["followers_per_friend"] = train_fe["followers_count"] / (train_fe["friends_count"] + 1)
test_fe["followers_per_friend"]  = test_fe["followers_count"] / (test_fe["friends_count"] + 1)

train_fe["statuses_per_day"] = train_fe["statuses_count"] / (train_fe["account_age_days"] + 1)
test_fe["statuses_per_day"]  = test_fe["statuses_count"] / (test_fe["account_age_days"] + 1)

# Screen name features
for df in [train_fe, test_fe]:
    df["screen_name_len"] = df["screen_name"].apply(safe_len)
    df["screen_name_digit_ratio"] = df["screen_name"].apply(frac_digits)
    df["screen_name_ends_with_digit"] = df["screen_name"].apply(ends_with_digit)

# Description features
for df in [train_fe, test_fe]:
    df["description_len"] = df["description"].apply(safe_len)
    df["description_word_count"] = df["description"].apply(word_count)
    df["description_avg_word_len"] = df["description"].apply(avg_word_len)
    df["description_special_char_ratio"] = df["description"].apply(special_char_ratio)
    df["has_url_in_description"] = df["description"].str.contains("http|www", case=False).astype(int)

# Time-based features
for df in [train_fe, test_fe]:
    dt = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    df["account_creation_hour"]  = dt.dt.hour.fillna(0).astype(int)
    df["account_creation_dow"]   = dt.dt.dayofweek.fillna(0).astype(int)
    df["account_creation_month"] = dt.dt.month.fillna(0).astype(int)

    # Cyclical encoding
    df["hour_sin"] = np.sin(2 * np.pi * df["account_creation_hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["account_creation_hour"] / 24)



In [22]:
#@title Text Embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

def tfidf_svd_embed(train_series, test_series, name, analyzer="word", ngram=(1,2), n_comp=64, max_feat=15000):
    all_text = pd.concat([train_series, test_series]).fillna("").astype(str)
    tfidf = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, max_features=max_feat)
    svd = TruncatedSVD(n_components=n_comp, random_state=42)

    tfidf_mat = tfidf.fit_transform(all_text)
    svd_mat = svd.fit_transform(tfidf_mat)

    df_svd_train = pd.DataFrame(svd_mat[:len(train_series)], columns=[f"svd_{name}_{i}" for i in range(n_comp)])
    df_svd_test  = pd.DataFrame(svd_mat[len(train_series):], columns=[f"svd_{name}_{i}" for i in range(n_comp)])
    return df_svd_train, df_svd_test

desc_tr, desc_te = tfidf_svd_embed(train_fe["description"], test_fe["description"], name="desc", analyzer="word", ngram=(1,3), n_comp=100)
loc_tr, loc_te   = tfidf_svd_embed(train_fe["location"], test_fe["location"], name="loc", analyzer="word", ngram=(1,2), n_comp=40)
sn_tr, sn_te     = tfidf_svd_embed(train_fe["screen_name"], test_fe["screen_name"], name="screen", analyzer="char", ngram=(3,5), n_comp=24)

# Merge back
train_fe = pd.concat([train_fe, desc_tr, loc_tr, sn_tr], axis=1)
test_fe  = pd.concat([test_fe,  desc_te, loc_te, sn_te], axis=1)


In [23]:
#@title HistGradientBoosting
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

drop_cols = {"target", "id", "screen_name", "description", "location", "lang", "created_at", "profile_image_url", "profile_background_image_url"}
features = [c for c in train_fe.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(train_fe[c])]

X = train_fe[features].values
y = train_fe["target"].values
X_test = test_fe[features].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(learning_rate=0.1, max_iter=300, random_state=42))
])

oof_preds = np.zeros(len(y))
fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    model.fit(X_tr, y_tr)
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds
    auc = roc_auc_score(y_val, val_preds)
    fold_aucs.append(auc)
    print(f"[Fold {fold}] AUC: {auc:.6f}")

print(f"\nOOF AUC: {roc_auc_score(y, oof_preds):.6f}")


[Fold 1] AUC: 0.933471
[Fold 2] AUC: 0.941934
[Fold 3] AUC: 0.940306
[Fold 4] AUC: 0.933861
[Fold 5] AUC: 0.935413

OOF AUC: 0.936882


In [25]:
#@title Helper Functions

import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier, log_evaluation, early_stopping

def scale_pos_weight_from_y(y):
    pos = np.sum(y == 1)
    neg = np.sum(y == 0)
    return neg / pos if pos > 0 else 1.0

def summarize(name, train_aucs, val_aucs, oof_auc):
    print(f"\n[{name}] Mean Train AUC: {np.mean(train_aucs):.6f}")
    print(f"[{name}] Mean Val AUC:   {np.mean(val_aucs):.6f}")
    print(f"[{name}] OOF AUC:        {oof_auc:.6f}")

In [26]:
#@title LGBM
def run_cv_lgbm(X, y, Xte, cv):
    oof = np.zeros(len(y), dtype=float)
    test_preds = np.zeros(Xte.shape[0], dtype=float)
    train_aucs, val_aucs = [], []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        spw = scale_pos_weight_from_y(ytr)

        lgbm = LGBMClassifier(
            objective="binary",
            n_estimators=10000,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.85,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            min_child_samples=30,
            random_state=42 + fold,
            n_jobs=-1,
            scale_pos_weight=spw,
            verbose=-1
        )

        lgbm.fit(
            Xtr, ytr,
            eval_set=[(Xtr, ytr), (Xva, yva)],
            eval_metric="auc",
            callbacks=[early_stopping(500), log_evaluation(100)]
        )

        va_pred = lgbm.predict_proba(Xva)[:, 1]
        oof[va_idx] = va_pred
        test_preds += lgbm.predict_proba(Xte)[:, 1] / cv.n_splits

        tr_pred = lgbm.predict_proba(Xtr)[:, 1]
        train_aucs.append(roc_auc_score(ytr, tr_pred))
        val_aucs.append(roc_auc_score(yva, va_pred))
        print(f"[LGB Fold {fold}] AUC: {val_aucs[-1]:.6f}")

    oof_auc = roc_auc_score(y, oof)
    summarize("LGB", train_aucs, val_aucs, oof_auc)
    return oof, test_preds

from sklearn.datasets import make_classification

# sample data
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    random_state=42
)
Xte, _ = make_classification(
    n_samples=500,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    random_state=99
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof, preds = run_cv_lgbm(X, y, Xte, cv)


Training until validation scores don't improve for 500 rounds
[100]	training's auc: 0.9982	training's binary_logloss: 0.162968	valid_1's auc: 0.971897	valid_1's binary_logloss: 0.252712
[200]	training's auc: 0.999987	training's binary_logloss: 0.0634587	valid_1's auc: 0.980198	valid_1's binary_logloss: 0.185133
[300]	training's auc: 1	training's binary_logloss: 0.0316787	valid_1's auc: 0.982198	valid_1's binary_logloss: 0.17106
[400]	training's auc: 1	training's binary_logloss: 0.0185822	valid_1's auc: 0.981998	valid_1's binary_logloss: 0.166779
[500]	training's auc: 1	training's binary_logloss: 0.0121627	valid_1's auc: 0.982598	valid_1's binary_logloss: 0.164697
[600]	training's auc: 1	training's binary_logloss: 0.00856713	valid_1's auc: 0.982998	valid_1's binary_logloss: 0.163255
[700]	training's auc: 1	training's binary_logloss: 0.00646643	valid_1's auc: 0.983498	valid_1's binary_logloss: 0.162862
[800]	training's auc: 1	training's binary_logloss: 0.00508734	valid_1's auc: 0.984398	

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 500 rounds
[100]	training's auc: 0.998269	training's binary_logloss: 0.166784	valid_1's auc: 0.976798	valid_1's binary_logloss: 0.237204
[200]	training's auc: 1	training's binary_logloss: 0.0667884	valid_1's auc: 0.977598	valid_1's binary_logloss: 0.181721
[300]	training's auc: 1	training's binary_logloss: 0.0332107	valid_1's auc: 0.978398	valid_1's binary_logloss: 0.17089
[400]	training's auc: 1	training's binary_logloss: 0.0189653	valid_1's auc: 0.978198	valid_1's binary_logloss: 0.170359
[500]	training's auc: 1	training's binary_logloss: 0.0122921	valid_1's auc: 0.977798	valid_1's binary_logloss: 0.172104
[600]	training's auc: 1	training's binary_logloss: 0.0086982	valid_1's auc: 0.978098	valid_1's binary_logloss: 0.173875
Early stopping, best iteration is:
[168]	training's auc: 0.999962	training's binary_logloss: 0.0872733	valid_1's auc: 0.978398	valid_1's binary_logloss: 0.190828
[LGB Fold 2] AUC: 0.978398
Training until validatio

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[200]	training's auc: 0.999994	training's binary_logloss: 0.0582516	valid_1's auc: 0.957496	valid_1's binary_logloss: 0.250756
[300]	training's auc: 1	training's binary_logloss: 0.0287543	valid_1's auc: 0.961696	valid_1's binary_logloss: 0.240561
[400]	training's auc: 1	training's binary_logloss: 0.0167471	valid_1's auc: 0.962596	valid_1's binary_logloss: 0.243591
[500]	training's auc: 1	training's binary_logloss: 0.0108955	valid_1's auc: 0.962596	valid_1's binary_logloss: 0.248374
[600]	training's auc: 1	training's binary_logloss: 0.00774351	valid_1's auc: 0.962796	valid_1's binary_logloss: 0.253236
[700]	training's auc: 1	training's binary_logloss: 0.00585832	valid_1's auc: 0.962996	valid_1's binary_logloss: 0.25866
Early stopping, best iteration is:
[267]	training's auc: 1	training's binary_logloss: 0.0356435	valid_1's auc: 0.961396	valid_1's binary_logloss: 0.23992
[LGB Fold 3] AUC: 0.961396
Training until validation scores don't improve for 500 rounds
[100]	training's auc: 0.99768

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[200]	training's auc: 0.999994	training's binary_logloss: 0.0655794	valid_1's auc: 0.983	valid_1's binary_logloss: 0.186684
[300]	training's auc: 1	training's binary_logloss: 0.0318173	valid_1's auc: 0.9835	valid_1's binary_logloss: 0.166903
[400]	training's auc: 1	training's binary_logloss: 0.018361	valid_1's auc: 0.984	valid_1's binary_logloss: 0.161303
[500]	training's auc: 1	training's binary_logloss: 0.0118915	valid_1's auc: 0.9841	valid_1's binary_logloss: 0.159627
[600]	training's auc: 1	training's binary_logloss: 0.00844577	valid_1's auc: 0.9846	valid_1's binary_logloss: 0.158619
[700]	training's auc: 1	training's binary_logloss: 0.00637043	valid_1's auc: 0.9842	valid_1's binary_logloss: 0.160868
[800]	training's auc: 1	training's binary_logloss: 0.00503693	valid_1's auc: 0.9844	valid_1's binary_logloss: 0.161041
[900]	training's auc: 1	training's binary_logloss: 0.00411992	valid_1's auc: 0.9845	valid_1's binary_logloss: 0.16273
[1000]	training's auc: 1	training's binary_loglos

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LGB Fold 4] AUC: 0.984600
Training until validation scores don't improve for 500 rounds
[100]	training's auc: 0.998406	training's binary_logloss: 0.164221	valid_1's auc: 0.9653	valid_1's binary_logloss: 0.280466
[200]	training's auc: 1	training's binary_logloss: 0.0633863	valid_1's auc: 0.9684	valid_1's binary_logloss: 0.233953
[300]	training's auc: 1	training's binary_logloss: 0.03093	valid_1's auc: 0.9699	valid_1's binary_logloss: 0.222971
[400]	training's auc: 1	training's binary_logloss: 0.0178744	valid_1's auc: 0.9692	valid_1's binary_logloss: 0.223041
[500]	training's auc: 1	training's binary_logloss: 0.0116229	valid_1's auc: 0.9687	valid_1's binary_logloss: 0.228157
[600]	training's auc: 1	training's binary_logloss: 0.00827151	valid_1's auc: 0.9682	valid_1's binary_logloss: 0.231908
[700]	training's auc: 1	training's binary_logloss: 0.00616325	valid_1's auc: 0.9685	valid_1's binary_logloss: 0.232983
[800]	training's auc: 1	training's binary_logloss: 0.00485909	valid_1's auc: 0.

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [27]:
#@title XGBoost
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb

from packaging import version
_XGB_SUPPORTS_ITER_RANGE = version.parse(xgb.__version__) >= version.parse("1.6.0")

def _predict(booster, dmat):
    if _XGB_SUPPORTS_ITER_RANGE:
        return booster.predict(dmat, iteration_range=(0, booster.best_iteration + 1))
    else:
        # for Google Colab older XGBoost
        return booster.predict(dmat, ntree_limit=booster.best_ntree_limit)

def run_cv_xgb(X, y, Xte, cv):
    oof = np.zeros(len(y), dtype=float)
    test_preds = np.zeros(Xte.shape[0], dtype=float)
    train_aucs, val_aucs = [], []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        spw = scale_pos_weight_from_y(ytr)

        dtr = xgb.DMatrix(Xtr, label=ytr)
        dva = xgb.DMatrix(Xva,  label=yva)
        dte = xgb.DMatrix(Xte)

        params = {
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "eta": 0.03,
            "max_depth": 6,
            "subsample": 0.85,
            "colsample_bytree": 0.8,
            "lambda": 1.0,
            "min_child_weight": 30,
            "tree_method": "hist",
            "seed": 42 + fold,
            "nthread": -1,
            "scale_pos_weight": spw,
            "verbosity": 0,
        }

        booster = xgb.train(
            params=params,
            dtrain=dtr,
            num_boost_round=10000,
            evals=[(dtr, "train"), (dva, "valid")],
            early_stopping_rounds=500,
            verbose_eval=False,
        )

        va_pred = _predict(booster, dva)
        oof[va_idx] = va_pred
        test_preds += _predict(booster, dte) / cv.n_splits

        tr_pred = _predict(booster, dtr)
        train_aucs.append(roc_auc_score(ytr, tr_pred))
        val_aucs.append(roc_auc_score(yva, va_pred))
        print(f"[XGB Fold {fold}] AUC: {val_aucs[-1]:.6f}")

    oof_auc = roc_auc_score(y, oof)
    summarize("XGB", train_aucs, val_aucs, oof_auc)
    return oof, test_preds

from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=10, n_redundant=5, random_state=42
)
Xte, _ = make_classification(
    n_samples=400, n_features=20, n_informative=10, n_redundant=5, random_state=7
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof, test_preds = run_cv_xgb(X, y, Xte, cv)


[XGB Fold 1] AUC: 0.942594
[XGB Fold 2] AUC: 0.953995
[XGB Fold 3] AUC: 0.926693
[XGB Fold 4] AUC: 0.939800
[XGB Fold 5] AUC: 0.946600

[XGB] Mean Train AUC: 0.967623
[XGB] Mean Val AUC:   0.941936
[XGB] OOF AUC:        0.941734


In [28]:
#@title CatBoost
!pip install catboost
from catboost import CatBoostClassifier

def run_cv_cat(X, y, Xte, cv):
    oof = np.zeros(len(y), dtype=float)
    test_preds = np.zeros(Xte.shape[0], dtype=float)
    train_aucs, val_aucs = [], []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        cat = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            iterations=10000,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=3.0,
            subsample=0.85,
            random_seed=42 + fold,
            verbose=False,
            early_stopping_rounds=500,
            allow_writing_files=False,
            auto_class_weights="Balanced"
        )

        cat.fit(Xtr, ytr, eval_set=(Xva, yva), verbose=False)

        va_pred = cat.predict_proba(Xva)[:, 1]
        oof[va_idx] = va_pred
        test_preds += cat.predict_proba(Xte)[:, 1] / cv.n_splits

        tr_pred = cat.predict_proba(Xtr)[:, 1]
        train_aucs.append(roc_auc_score(ytr, tr_pred))
        val_aucs.append(roc_auc_score(yva, va_pred))
        print(f"[CAT Fold {fold}] AUC: {val_aucs[-1]:.6f}")

    oof_auc = roc_auc_score(y, oof)
    summarize("CAT", train_aucs, val_aucs, oof_auc)
    return oof, test_preds

#Cross Validation

from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=42)
Xte, _ = make_classification(n_samples=400, n_features=20, n_informative=10, random_state=99)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof, test_preds = run_cv_cat(X, y, Xte, cv)


[CAT Fold 1] AUC: 0.992599
[CAT Fold 2] AUC: 0.987600
[CAT Fold 3] AUC: 0.977600
[CAT Fold 4] AUC: 0.989000
[CAT Fold 5] AUC: 0.994000

[CAT] Mean Train AUC: 1.000000
[CAT] Mean Val AUC:   0.988160
[CAT] OOF AUC:        0.987144


In [ ]:
# @title Blending: LGBM + XGB + CAT
!pip -q install lightgbm xgboost catboost scikit-learn --upgrade

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import xgboost as xgb
from catboost import CatBoostClassifier

df_tr = train_fe.copy()
df_te = test_fe.copy()

drop_cols = {
    "target", "id", "screen_name", "description", "location", "lang",
    "created_at", "profile_image_url", "profile_background_image_url"
}
num_cols = [c for c in df_tr.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(df_tr[c])]

X = df_tr[num_cols].values
y = df_tr["target"].values
X_test = df_te[num_cols].values

# Basic imputing
imp = SimpleImputer(strategy="median")
X = imp.fit_transform(X)
X_test = imp.transform(X_test)

# Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def scale_pos_weight_from_y(y_vec):
    pos = (y_vec == 1).sum()
    neg = (y_vec == 0).sum()
    return (neg / max(pos, 1))


# 1) LGBM
def run_cv_lgbm(X, y, Xte, cv, base_seed=42):
    oof = np.zeros(len(y), dtype=float)
    te = np.zeros(Xte.shape[0], dtype=float)
    tr_aucs, va_aucs = [], []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        spw = scale_pos_weight_from_y(ytr)
        clf = LGBMClassifier(
            objective="binary",
            n_estimators=10000,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.85,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            min_child_samples=40,
            random_state=base_seed + fold,
            n_jobs=-1,
            scale_pos_weight=spw,
            verbose=-1,
        )
        clf.fit(
            Xtr, ytr,
            eval_set=[(Xtr, ytr), (Xva, yva)],
            eval_metric="auc",
            callbacks=[early_stopping(500), log_evaluation(100)]
        )
        va_pred = clf.predict_proba(Xva)[:, 1]
        tr_pred = clf.predict_proba(Xtr)[:, 1]

        oof[va_idx] = va_pred
        te += clf.predict_proba(Xte)[:, 1] / cv.n_splits

        tr_aucs.append(roc_auc_score(ytr, tr_pred))
        va_aucs.append(roc_auc_score(yva, va_pred))
        print(f"[LGB Fold {fold}] AUC: {va_aucs[-1]:.6f}")

    oof_auc = roc_auc_score(y, oof)
    print(f"[LGB] Mean Val AUC: {np.mean(va_aucs):.6f} | OOF AUC: {oof_auc:.6f}")
    return {"name": "lgb", "oof": oof, "test": te, "fold_aucs": va_aucs, "oof_auc": oof_auc}

# 2) XGBoost
from packaging import version
_XGB_SUPPORTS_ITER_RANGE = version.parse(xgb.__version__) >= version.parse("1.6.0")
def _xgb_predict(booster, dmat):
    if _XGB_SUPPORTS_ITER_RANGE:
        return booster.predict(dmat, iteration_range=(0, booster.best_iteration + 1))
    else:
        return booster.predict(dmat, ntree_limit=booster.best_ntree_limit)

def run_cv_xgb(X, y, Xte, cv, base_seed=42):
    oof = np.zeros(len(y), dtype=float)
    te = np.zeros(Xte.shape[0], dtype=float)
    tr_aucs, va_aucs = [], []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        spw = scale_pos_weight_from_y(ytr)
        dtr, dva, dte = xgb.DMatrix(Xtr, label=ytr), xgb.DMatrix(Xva, label=yva), xgb.DMatrix(Xte)

        params = {
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "eta": 0.03,
            "max_depth": 6,
            "subsample": 0.85,
            "colsample_bytree": 0.8,
            "lambda": 1.0,
            "min_child_weight": 30,
            "tree_method": "hist",
            "seed": base_seed + fold,
            "nthread": -1,
            "scale_pos_weight": spw,
            "verbosity": 0,
        }
        booster = xgb.train(
            params=params,
            dtrain=dtr,
            num_boost_round=10000,
            evals=[(dtr, "train"), (dva, "valid")],
            early_stopping_rounds=500,
            verbose_eval=False,
        )

        va_pred = _xgb_predict(booster, dva)
        tr_pred = _xgb_predict(booster, dtr)
        oof[va_idx] = va_pred
        te += _xgb_predict(booster, dte) / cv.n_splits

        tr_aucs.append(roc_auc_score(ytr, tr_pred))
        va_aucs.append(roc_auc_score(yva, va_pred))
        print(f"[XGB Fold {fold}] AUC: {va_aucs[-1]:.6f}")

    oof_auc = roc_auc_score(y, oof)
    print(f"[XGB] Mean Val AUC: {np.mean(va_aucs):.6f} | OOF AUC: {oof_auc:.6f}")
    return {"name": "xgb", "oof": oof, "test": te, "fold_aucs": va_aucs, "oof_auc": oof_auc}

# 3) Catboost
def run_cv_cat(X, y, Xte, cv, base_seed=42):
    oof = np.zeros(len(y), dtype=float)
    te = np.zeros(Xte.shape[0], dtype=float)
    tr_aucs, va_aucs = [], []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        cat = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            iterations=10000,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=3.0,
            subsample=0.85,
            random_seed=base_seed + fold,
            verbose=False,
            early_stopping_rounds=500,
            allow_writing_files=False,
            auto_class_weights="Balanced"
        )
        cat.fit(Xtr, ytr, eval_set=(Xva, yva), verbose=False)

        va_pred = cat.predict_proba(Xva)[:, 1]
        tr_pred = cat.predict_proba(Xtr)[:, 1]
        oof[va_idx] = va_pred
        te += cat.predict_proba(Xte)[:, 1] / cv.n_splits

        tr_aucs.append(roc_auc_score(ytr, tr_pred))
        va_aucs.append(roc_auc_score(yva, va_pred))
        print(f"[CAT Fold {fold}] AUC: {va_aucs[-1]:.6f}")

    oof_auc = roc_auc_score(y, oof)
    print(f"[CAT] Mean Val AUC: {np.mean(va_aucs):.6f} | OOF AUC: {oof_auc:.6f}")
    return {"name": "cat", "oof": oof, "test": te, "fold_aucs": va_aucs, "oof_auc": oof_auc}

# train all
res_lgb = run_cv_lgbm(X, y, X_test, cv, base_seed=42)
res_xgb = run_cv_xgb(X, y, X_test, cv, base_seed=4242)
res_cat = run_cv_cat(X, y, X_test, cv, base_seed=2025)

# Collect predictions
oofs  = np.vstack([res_lgb["oof"], res_xgb["oof"], res_cat["oof"]])   # shape: (3, n_train)
tests = np.vstack([res_lgb["test"], res_xgb["test"], res_cat["test"]]) # shape: (3, n_test)

# 4)Bbelnding

def auc(y_true, y_pred): return roc_auc_score(y_true, y_pred)

# Mean
oof_mean = oofs.mean(axis=0)
test_mean = tests.mean(axis=0)
auc_mean = auc(y, oof_mean)

# AUC-weighted mean
w = np.array([np.mean(res_lgb["fold_aucs"]),
              np.mean(res_xgb["fold_aucs"]),
              np.mean(res_cat["fold_aucs"])])
w = np.clip(w, 1e-6, None)
w = w / w.sum()
oof_w = (oofs.T @ w).ravel()
test_w = (tests.T @ w).ravel()
auc_w = auc(y, oof_w)

# Rank-average
def rank_avg(arr2d):
    ranks = np.vstack([pd.Series(a).rank(method="average").values for a in arr2d])
    return ranks.mean(axis=0) / ranks.shape[1]
oof_rank = rank_avg(oofs)
test_rank = rank_avg(tests)
auc_rank = auc(y, oof_rank)

# Logistic stacker on OOFs
meta_X = oofs.T  # shape (n_train, 3)
meta_te = tests.T  # shape (n_test, 3)

meta_oof = np.zeros(len(y))
meta_test_blend = np.zeros(X_test.shape[0])

meta_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
for tr_idx, va_idx in meta_cv.split(meta_X, y):
    Xtr_m, Xva_m = meta_X[tr_idx], meta_X[va_idx]
    ytr_m, yva_m = y[tr_idx], y[va_idx]
    m = LogisticRegression(
        solver="lbfgs",
        max_iter=200,
        C=1.0,
        class_weight=None,
        n_jobs=None
    )
    m.fit(Xtr_m, ytr_m)
    meta_oof[va_idx] = m.predict_proba(Xva_m)[:, 1]
    meta_test_blend += m.predict_proba(meta_te)[:, 1] / meta_cv.n_splits

auc_meta = auc(y, meta_oof)

print("\n=== BLEND AUCs (OOF) ===")
print(f"Simple mean:     {auc_mean:.6f}")
print(f"AUC-weighted:    {auc_w:.6f} (weights={w.round(4).tolist()})")
print(f"Rank-average:    {auc_rank:.6f}")
print(f"Logit stacker:   {auc_meta:.6f}")

# Choose the best blend by OOF AUC
blend_names = ["mean", "auc_weighted", "rank_avg", "stacker"]
blend_oofs  = [oof_mean, oof_w, oof_rank, meta_oof]
blend_tests = [test_mean, test_w, test_rank, meta_test_blend]
blend_aucs  = [auc_mean, auc_w, auc_rank, auc_meta]

best_idx = int(np.argmax(blend_aucs))
best_name = blend_names[best_idx]
best_auc  = blend_aucs[best_idx]
best_test = blend_tests[best_idx]
print(f"\n>> Best blend = {best_name} | OOF AUC = {best_auc:.6f}")

# Submission Format

submission = pd.DataFrame({
    "index": np.arange(len(best_test)),   # if the competition expects row index
    "target": best_test
})
print(submission.head())


Training until validation scores don't improve for 500 rounds


In [ ]:
# @title Advanced Modeling with LightGBM with micro-bagging
!pip -q install numpy pandas scikit-learn lightgbm xgboost catboost

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer

if 'train_fe' in globals() and 'test_fe' in globals():
    tr_df = train_fe.copy()
    te_df = test_fe.copy()
else:
    tr_df = train.copy()
    te_df = test.copy()

drop_cols = {
    "target", "index", "id", "screen_name", "created_at",
    "profile_image_url", "profile_background_image_url",
    "description", "location", "lang", "hour_bin", "lang_mod_te"
}

cand_cols = tr_df.select_dtypes(include=['number', 'bool']).columns.tolist()
feature_cols = [c for c in cand_cols if c not in drop_cols]

X_all = tr_df[feature_cols].astype(float).values
y_all = tr_df["target"].astype(int).values
X_test = te_df[feature_cols].astype(float).values

if "index" in te_df.columns:
    test_index = te_df["index"].values
else:
    test_index = np.arange(len(te_df))

# safety impute
imputer = SimpleImputer(strategy="median")
X_all = imputer.fit_transform(X_all)
X_test = imputer.transform(X_test)

print(f"Selected {len(feature_cols)} features.")

# cross validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def scale_pos_weight_from_y(y):
    pos = np.sum(y)
    neg = len(y) - pos
    return float(neg / pos) if pos > 0 else 1.0

def summarize(label, train_aucs, val_aucs, oof_auc):
    print(f"\n[{label}] Mean Train AUC: {np.mean(train_aucs):.4f} | Mean Val AUC: {np.mean(val_aucs):.4f}")
    print(f"[{label}] OOF AUC: {oof_auc:.4f}")

# LightGBM with optional micro-bagging
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

BASE_LGB_PARAMS = dict(
    objective="binary",
    n_estimators=10000,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.85,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_samples=30,
    n_jobs=-1,
    verbose=-1
)

def run_cv_lgbm(X, y, Xte, cv, params):
    oof = np.zeros(len(y), dtype=float)
    test_preds = np.zeros(Xte.shape[0], dtype=float)
    train_aucs, val_aucs = []

    # initialize to avoid UnboundLocalError if list unpacking fails
    train_aucs = []
    val_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        spw = scale_pos_weight_from_y(ytr)
        p = dict(params); p.update(dict(random_state=42 + fold, scale_pos_weight=spw))

        model = LGBMClassifier(**p)
        model.fit(
            Xtr, ytr,
            eval_set=[(Xtr, ytr), (Xva, yva)],
            eval_metric="auc",
            callbacks=[early_stopping(500), log_evaluation(0)]
        )

        va_pred = model.predict_proba(Xva)[:, 1]
        oof[va_idx] = va_pred
        test_preds += model.predict_proba(Xte)[:, 1] / cv.n_splits

        tr_pred = model.predict_proba(Xtr)[:, 1]
        train_aucs.append(roc_auc_score(ytr, tr_pred))
        val_aucs.append(roc_auc_score(yva, va_pred))
        print(f"[LGB Fold {fold}] AUC: {val_aucs[-1]:.6f}")

    oof_auc = roc_auc_score(y, oof)
    summarize("LGB", train_aucs, val_aucs, oof_auc)
    return oof, test_preds

def lgb_bag(X, y, Xte, cv, base_params, seeds=(11,22,33,44,55)):
    """Micro-bagging across seeds; returns bagged OOF & test preds."""
    bag_oof = np.zeros(len(y), dtype=float)
    bag_test = np.zeros(Xte.shape[0], dtype=float)
    fold_aucs_all = []

    for seed in seeds:
        oof = np.zeros(len(y), dtype=float)
        test_preds = np.zeros(Xte.shape[0], dtype=float)
        train_aucs, val_aucs = [], []

        for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
            Xtr, Xva = X[tr_idx], X[va_idx]
            ytr, yva = y[tr_idx], y[va_idx]

            spw = scale_pos_weight_from_y(ytr)
            p = dict(base_params); p.update(dict(random_state=seed + fold, scale_pos_weight=spw))

            model = LGBMClassifier(**p)
            model.fit(
                Xtr, ytr,
                eval_set=[(Xtr, ytr), (Xva, yva)],
                eval_metric="auc",
                callbacks=[early_stopping(400), log_evaluation(0)]
            )

            va_pred = model.predict_proba(Xva)[:, 1]
            oof[va_idx] = va_pred
            test_preds += model.predict_proba(Xte)[:, 1] / cv.n_splits

            tr_pred = model.predict_proba(Xtr)[:, 1]
            train_aucs.append(roc_auc_score(ytr, tr_pred))
            val_aucs.append(roc_auc_score(yva, va_pred))

        fold_aucs_all.append(np.mean(val_aucs))
        bag_oof += oof / len(seeds)
        bag_test += test_preds / len(seeds)

    print(f"[LGB bag] Mean fold AUC across seeds: {np.mean(fold_aucs_all):.6f}")
    print(f"[LGB bag] OOF AUC: {roc_auc_score(y, bag_oof):.6f}")
    return bag_oof, bag_test

USE_LGB_BAGGING = True


# 4) XGBoost
import xgboost as xgb

def run_cv_xgb(X, y, Xte, cv):
    oof = np.zeros(len(y), dtype=float)
    test_preds = np.zeros(Xte.shape[0], dtype=float)
    train_aucs, val_aucs = [], []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        spw = scale_pos_weight_from_y(ytr)

        dtr = xgb.DMatrix(Xtr, label=ytr)
        dva = xgb.DMatrix(Xva,  label=yva)
        dte = xgb.DMatrix(Xte)

        params = {
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "eta": 0.03,
            "max_depth": 6,
            "subsample": 0.85,
            "colsample_bytree": 0.8,
            "lambda": 1.0,
            "min_child_weight": 30,
            "tree_method": "hist",
            "seed": 42 + fold,
            "nthread": -1,
            "scale_pos_weight": spw,
            "verbosity": 0
        }

        booster = xgb.train(
            params=params,
            dtrain=dtr,
            num_boost_round=10000,
            evals=[(dtr, "train"), (dva, "valid")],
            early_stopping_rounds=500,
            verbose_eval=False
        )

        va_pred = booster.predict(dva, iteration_range=(0, booster.best_iteration + 1))
        oof[va_idx] = va_pred
        test_preds += booster.predict(dte, iteration_range=(0, booster.best_iteration + 1)) / cv.n_splits

        tr_pred = booster.predict(dtr, iteration_range=(0, booster.best_iteration + 1))
        train_aucs.append(roc_auc_score(ytr, tr_pred))
        val_aucs.append(roc_auc_score(yva, va_pred))
        print(f"[XGB Fold {fold}] AUC: {val_aucs[-1]:.6f}")

    oof_auc = roc_auc_score(y, oof)
    summarize("XGB", train_aucs, val_aucs, oof_auc)
    return oof, test_preds

# CatBoost
CATBOOST_AVAILABLE = True
try:
    from catboost import CatBoostClassifier
except Exception:
    CATBOOST_AVAILABLE = False
    print("CatBoost not available; skipping CatBoost model.")

def run_cv_cat(X, y, Xte, cv):
    if not CATBOOST_AVAILABLE:
        return np.zeros(len(y), dtype=float), np.zeros(Xte.shape[0], dtype=float)

    oof = np.zeros(len(y), dtype=float)
    test_preds = np.zeros(Xte.shape[0], dtype=float)
    train_aucs, val_aucs = [], []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        cat = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            iterations=10000,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=3.0,
            subsample=0.85,
            random_seed=42 + fold,
            early_stopping_rounds=500,
            allow_writing_files=False,
            auto_class_weights="Balanced",
            verbose=False
        )

        cat.fit(Xtr, ytr, eval_set=(Xva, yva), verbose=False)

        va_pred = cat.predict_proba(Xva)[:, 1]
        oof[va_idx] = va_pred
        test_preds += cat.predict_proba(Xte)[:, 1] / cv.n_splits

        tr_pred = cat.predict_proba(Xtr)[:, 1]
        train_aucs.append(roc_auc_score(ytr, tr_pred))
        val_aucs.append(roc_auc_score(yva, va_pred))
        print(f"[CAT Fold {fold}] AUC: {val_aucs[-1]:.6f}")

    oof_auc = roc_auc_score(y, oof)
    summarize("CAT", train_aucs, val_aucs, oof_auc)
    return oof, test_preds


# 6) Run & Blend (optimized weights)

if USE_LGB_BAGGING:
    lgb_oof, lgb_test = lgb_bag(X_all, y_all, X_test, skf, BASE_LGB_PARAMS, seeds=(11,22,33,44,55))
else:
    lgb_oof, lgb_test = run_cv_lgbm(X_all, y_all, X_test, skf, BASE_LGB_PARAMS)

xgb_oof, xgb_test = run_cv_xgb(X_all, y_all, X_test, skf)
cat_oof, cat_test = run_cv_cat(X_all, y_all, X_test, skf)

print("\n[Single-model OOF]")
print(f"LGB  OOF AUC: {roc_auc_score(y_all, lgb_oof):.6f}")
print(f"XGB  OOF AUC: {roc_auc_score(y_all, xgb_oof):.6f}")
if CATBOOST_AVAILABLE:
    print(f"CAT  OOF AUC: {roc_auc_score(y_all, cat_oof):.6f}")

# Use your optimized weights from search: (0.55, 0.20, 0.25)
w_lgb, w_xgb, w_cat = 0.55, 0.20, 0.25
blend_oof  = w_lgb*lgb_oof + w_xgb*xgb_oof + w_cat*cat_oof
blend_test = w_lgb*lgb_test + w_xgb*xgb_test + w_cat*cat_test

print(f"\n[Weighted Blend] OOF AUC: {roc_auc_score(y_all, blend_oof):.6f}")

# 7) Submission
final_test = np.clip(blend_test, 1e-5, 1-1e-5)  # clip to avoid exact 0/1
submission = pd.DataFrame({"index": test_index, "target": final_test})
print(submission.head())

# Save CSV
submission.to_csv("submission_advanced_blendopt.csv", index=False)
print("\nSaved: submission_advanced_v3.csv")


In [ ]:
# @title Optuna + LightGBM: self-contained tuning
!pip -q install optuna lightgbm

import numpy as np
import pandas as pd
import optuna, inspect, collections
from optuna.trial import TrialState
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

if 'train_fe' in globals() and 'test_fe' in globals():
    tr_df = train_fe.copy()
    te_df = test_fe.copy()
else:
    tr_df = train.copy()
    te_df = test.copy()

drop_cols = {
    "target", "index", "id", "screen_name", "created_at",
    "profile_image_url", "profile_background_image_url",
    "description", "location", "lang", "hour_bin", "lang_mod_te"
}
cand_cols = tr_df.select_dtypes(include=["number", "bool"]).columns.tolist()
feature_cols = [c for c in cand_cols if c not in drop_cols]

X_all = tr_df[feature_cols].astype(float).values
y_all = tr_df["target"].astype(int).values
X_test = te_df[feature_cols].astype(float).values
test_index = (te_df["index"].values if "index" in te_df.columns else np.arange(len(te_df)))

imputer = SimpleImputer(strategy="median")
X_all = imputer.fit_transform(X_all)
X_test = imputer.transform(X_test)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def scale_pos_weight_from_y(y):
    pos = float(np.sum(y))
    neg = float(len(y) - np.sum(y))
    return (neg / pos) if pos > 0 else 1.0

print(f"[Optuna] Using {len(feature_cols)} features.")

# Function
def objective(trial: optuna.Trial) -> float:
    params = {
        "objective": "binary",
        "boosting_type": "gbdt",
        "n_estimators": 10000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 255, step=2),
        "max_depth": trial.suggest_categorical("max_depth", [-1, 4, 5, 6, 7, 8, 9, 10, 12]),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1,  # only effective if subsample < 1.0
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 100.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        "n_jobs": -1,
        "verbose": -1,
        "random_state": 777,
    }

    # Keep leaves consistent with depth if depth is bounded
    if params["max_depth"] not in (-1, None):
        max_leaves = 2 ** params["max_depth"]
        if params["num_leaves"] > max_leaves:
            params["num_leaves"] = max(31, max_leaves)

    oof = np.zeros_like(y_all, dtype=float)
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_all), 1):
        Xtr, Xva = X_all[tr_idx], X_all[va_idx]
        ytr, yva = y_all[tr_idx], y_all[va_idx]

        spw = scale_pos_weight_from_y(ytr)
        p = dict(params)
        p["scale_pos_weight"] = spw
        p["random_state"] = 1000 + fold

        model = LGBMClassifier(**p)
        model.fit(
            Xtr, ytr,
            eval_set=[(Xtr, ytr), (Xva, yva)],
            eval_metric="auc",
            callbacks=[early_stopping(400), log_evaluation(0)]
        )

        va_pred = model.predict_proba(Xva)[:, 1]
        oof[va_idx] = va_pred
        fold_auc = roc_auc_score(yva, va_pred)
        fold_aucs.append(fold_auc)

        # pruning hook
        trial.report(np.mean(fold_aucs), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return roc_auc_score(y_all, oof)

# Study: version-safe sampler/pruner
try:
    sampler = optuna.samplers.TPESampler(seed=42, multivariate=True, group=True)
except TypeError:
    sampler = optuna.samplers.TPESampler(seed=42, multivariate=True)

MedianPruner = optuna.pruners.MedianPruner
sig = inspect.signature(MedianPruner)
if "n_startup_trials" in sig.parameters:
    pruner = MedianPruner(n_startup_trials=10)
elif "n_warmup_steps" in sig.parameters:
    pruner = MedianPruner(n_warmup_steps=10)
else:
    pruner = MedianPruner()

study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

N_TRIALS = 60          # increase for better search
TIMEOUT_SEC = None     # or set a limit, e.g., 3600
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, show_progress_bar=True)

# Safe reporting
states = [t.state for t in study.trials]
print("Trial states:", collections.Counter(states))

if any(s == TrialState.COMPLETE for s in states):
    print("\nBest OOF AUC:", study.best_value)
    print("Best params:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")
    best_params = {
        "objective": "binary",
        "boosting_type": "gbdt",
        "n_estimators": 10000,
        "n_jobs": -1,
        "verbose": -1,
        **study.best_params,
    }
else:
    print("\nNo trials completed. Retrying a short run with NopPruner (no pruning)...")
    study = optuna.create_study(direction="maximize", sampler=sampler,
                                pruner=optuna.pruners.NopPruner())
    study.optimize(objective, n_trials=10, timeout=None, show_progress_bar=True)
    states = [t.state for t in study.trials]
    print("Trial states (nop):", collections.Counter(states))
    if any(s == TrialState.COMPLETE for s in states):
        print("\nBest OOF AUC (nop):", study.best_value)
        print("Best params (nop):")
        for k, v in study.best_params.items():
            print(f"  {k}: {v}")
        best_params = {
            "objective": "binary",
            "boosting_type": "gbdt",
            "n_estimators": 10000,
            "n_jobs": -1,
            "verbose": -1,
            **study.best_params,
        }
    else:
        raise RuntimeError("Optuna: still no completed trials. Check objective for errors/pruning too early.")

#Refit CV with best params and create submission
if 'best_params' in globals():
    oof = np.zeros_like(y_all, dtype=float)
    test_preds = np.zeros(X_test.shape[0], dtype=float)
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_all), 1):
        Xtr, Xva = X_all[tr_idx], X_all[va_idx]
        ytr, yva = y_all[tr_idx], y_all[va_idx]

        spw = scale_pos_weight_from_y(ytr)
        p = dict(best_params)
        p["scale_pos_weight"] = spw
        p["random_state"] = 4242 + fold

        model = LGBMClassifier(**p)
        model.fit(
            Xtr, ytr,
            eval_set=[(Xtr, ytr), (Xva, yva)],
            eval_metric="auc",
            callbacks=[early_stopping(500), log_evaluation(0)]
        )

        va_pred = model.predict_proba(Xva)[:, 1]
        oof[va_idx] = va_pred
        fold_aucs.append(roc_auc_score(yva, va_pred))
        test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

    oof_auc = roc_auc_score(y_all, oof)
    print(f"\n[Refit with best params] Mean fold AUC: {np.mean(fold_aucs):.6f}")
    print(f"[Refit with best params] OOF AUC: {oof_auc:.6f}")

    submission = pd.DataFrame({
        "index": test_index,
        "target": np.clip(test_preds, 1e-5, 1-1e-5)
    })
    submission.to_csv("submission_lgb_optuna.csv", index=False)
    print("Saved: submission_lgb_optuna.csv")
